In [22]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
import os
os.chdir(os.path.abspath(os.path.join(os.getcwd(), '..', '..')))


# Conversion and loading UVV files

In [ ]:
import glob
import os
import re
import csv
import pandas as pd

# Zweiter Versuch Metadaten und Tokenizing errors auf einmal zu lösen

In [65]:
import os
import glob
import re

RAW_DIR = 'data/raw_mrdga/'
OUT_DIR = 'data/raw_mrdga/neuer_trial'
MODEL_FILE = 'data/model_data/20240229_UVV_ALLA_BLPHI0001_1.csv'
# model metadata lines (1-based 604-637) -> 0-based slice [603:637]
MODEL_META_SLICE = slice(603, 637)

os.makedirs(OUT_DIR, exist_ok=True)

num_re = re.compile(r'^[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?$')

# read model metadata once
model_meta = []
if os.path.exists(MODEL_FILE):
    with open(MODEL_FILE, 'r', encoding='utf-8', errors='replace') as mf:
        mlines = mf.readlines()
    model_meta = mlines[MODEL_META_SLICE]

files = glob.glob(os.path.join(RAW_DIR, '*.csv'))

def is_number_string(s):
    return bool(num_re.match(s.strip()))

def first_col_value(line):
    parts = line.split(',', 1)
    if not parts:
        return None
    f = parts[0].strip()
    try:
        return float(f)
    except Exception:
        return None

def cut_to_collection_time(lines):
    # Return lines starting from the first line that begins with "Collection Time:"
    for i, ln in enumerate(lines):
        if ln.lstrip().lower().startswith('collection time:'):
            return lines[i:]
    return lines

def sanitize_meta_block(lines, header='spot1', replace_spot1=False):
    # Ensure metadata block begins with an empty first-column line (index 603),
    # then a header line "header," and then the header again on the next line.
    # Remove any existing leading empty/header lines from `lines` to avoid duplication,
    # strip trailing commas/spaces from metadata lines, and optionally replace 'spot1'.
    out = ['\n']
    header = (header or 'spot1').strip()
    header_line_comma = header + ',\n'
    header_line_plain = header + '\n'

    # First, cut incoming lines to start at "Collection Time:" if present.
    lines = cut_to_collection_time(lines)

    # Normalize incoming lines: remove leading empty lines and leading header-like lines
    start = 0
    while start < len(lines):
        s = lines[start].strip()
        # stop skipping when we find a non-empty and non-header line
        if not s:
            start += 1
            continue
        # if the line's first column equals header or 'spot1' (with or without trailing comma), skip it
        first_col = lines[start].split(',', 1)[0].strip()
        if first_col.lower() in (header.lower(), 'spot1'):
            start += 1
            continue
        break

    meta_lines = []
    for ln in lines[start:]:
        content = ln.rstrip('\n')
        # remove trailing commas/spaces for metadata
        content = re.sub(r',\s*$', '', content)
        if replace_spot1:
            # replace exact occurrence of 'spot1' in first column and elsewhere
            content = re.sub(r'\bspot1\b', header, content)
        meta_lines.append(content + '\n')

    # build block: empty line already added, then two header lines, then remaining metadata
    out.append(header_line_comma)
    out.append(header_line_plain)
    out.extend(meta_lines)
    return out

for fp in files:
    try:
        with open(fp, 'r', encoding='utf-8', errors='replace') as f:
            lines = f.readlines()
        # find first index where first column is between 199.5 and 200.5
        split_idx = None
        for i, ln in enumerate(lines):
            val = first_col_value(ln)
            if val is not None and 199.5 < val < 200.5:
                split_idx = i
                break

        out_lines = []
        if split_idx is None:
            # no split point found: copy whole file (but normalize line endings)
            out_lines = [ln if ln.endswith('\n') else ln + '\n' for ln in lines]
        else:
            # determine header from first column of original first line
            first_field = lines[0].rstrip('\n').split(',')[0].strip() or 'spot1'

            # special handling for first line: keep only first column, then two commas
            out_lines.append(first_field + ',,\n')

            # keep data lines from 1..split_idx, ensure each ends with a single comma
            for i in range(1, split_idx + 1):
                ln = lines[i].rstrip('\n').rstrip()
                # remove existing trailing commas/spaces, then add exactly one comma
                ln = re.sub(r',+\s*$', '', ln)
                out_lines.append(ln + ',\n')

            # decide how to append the remainder (metadata)
            remainder = lines[split_idx + 1 :]

            #
            col_count = None
            for r in remainder:
                rstr = r.strip()
                if not rstr:
                    continue
                col_count = len(rstr.split(','))
                break

            if col_count is None:
                # no remainder lines -> use model_meta if available, otherwise add minimal metadata block
                if model_meta:
                    out_lines.extend(sanitize_meta_block(model_meta, header=first_field, replace_spot1=True))
                else:
                    out_lines.extend(sanitize_meta_block([], header=first_field, replace_spot1=False))
            elif col_count <= 3:
                # append sanitized remainder but ensure metadata starts with an empty line and header lines
                # sanitize_meta_block will now cut the remainder to start at "Collection Time:" if present
                out_lines.extend(sanitize_meta_block(remainder, header=first_field, replace_spot1=False))
            else:
                # use model metadata block (fallback to original remainder if model metadata missing)
                if model_meta:
                    out_lines.extend(sanitize_meta_block(model_meta, header=first_field, replace_spot1=True))
                else:
                    out_lines.extend(sanitize_meta_block(remainder, header=first_field, replace_spot1=False))

        out_path = os.path.join(OUT_DIR, os.path.basename(fp))
        with open(out_path, 'w', encoding='utf-8') as outf:
            outf.writelines(out_lines)
        print(f"Wrote: {out_path} (orig: {os.path.basename(fp)})")
    except Exception as e:
        print(f"Failed {fp}: {e}")

Wrote: ../../data/raw_mrdga/neuer_trial\20240201_UVV_ALLA_LN46o0001_1.csv (orig: 20240201_UVV_ALLA_LN46o0001_1.csv)
Wrote: ../../data/raw_mrdga/neuer_trial\20240201_UVV_ALLA_LN46o0001_2.csv (orig: 20240201_UVV_ALLA_LN46o0001_2.csv)
Wrote: ../../data/raw_mrdga/neuer_trial\20240201_UVV_ALLA_LN46o0001_3.csv (orig: 20240201_UVV_ALLA_LN46o0001_3.csv)
Wrote: ../../data/raw_mrdga/neuer_trial\20240201_UVV_ALLA_LN46o0001_4.csv (orig: 20240201_UVV_ALLA_LN46o0001_4.csv)
Wrote: ../../data/raw_mrdga/neuer_trial\20240201_UVV_ALLA_LN46o0001_5.csv (orig: 20240201_UVV_ALLA_LN46o0001_5.csv)
Wrote: ../../data/raw_mrdga/neuer_trial\20240201_UVV_ALLA_LN46o0001_6.csv (orig: 20240201_UVV_ALLA_LN46o0001_6.csv)
Wrote: ../../data/raw_mrdga/neuer_trial\20240201_UVV_ALLA_LN46o0001_7.csv (orig: 20240201_UVV_ALLA_LN46o0001_7.csv)
Wrote: ../../data/raw_mrdga/neuer_trial\20240201_UVV_ALLA_LN46o0002_1.csv (orig: 20240201_UVV_ALLA_LN46o0002_1.csv)
Wrote: ../../data/raw_mrdga/neuer_trial\20240201_UVV_ALLA_LN46o0002_2.cs

In [ ]:
# check CSV readability in 'data/raw_mrdga/neuer_trial'
target_dir = 'data/raw_mrdga/neuer_trial'
files = glob.glob(os.path.join(target_dir, '*.csv'))

pandas_errors = []   # files pandas could not parse (pandas exception)
csv_fallback_errors = []  # files the csv module also fails to read (likely corrupt)
ok_files = []

for fp in files:
    try:
        # try pandas auto-sep parsing (engine='python' + sep=None uses the C engine's sniffing fallback)
        pd.read_csv(fp, sep=None, engine='python', encoding='utf-8', on_bad_lines='error')
        ok_files.append(fp)
    except Exception as e_pandas:
        # fallback: try Python csv reader to see if file is fundamentally unreadable
        try:
            with open(fp, 'r', encoding='utf-8', errors='replace') as f:
                for _ in csv.reader(f):
                    pass
            pandas_errors.append((fp, repr(e_pandas)))
        except Exception as e_csv:
            csv_fallback_errors.append((fp, repr(e_csv)))

errored_files = sorted({p for p, _ in pandas_errors} | {p for p, _ in csv_fallback_errors})

# summary
print(f"Total files checked: {len(files)}")
print(f"OK (pandas read): {len(ok_files)}")
print(f"Pandas-only errors: {pandas_errors}")
print(f"CSV-module errors (likely unreadable): {len(csv_fallback_errors)}")

# expose lists in the notebook namespace
ok_files, pandas_errors, csv_fallback_errors, errored_files

# files that had wrong metadata appended previously
# ('data/raw_mrdga/neuer_trial\\20240201_UVV_ALLA_LN46o0004_1.csv', "ParserError('Expected 3 fields in line 608, saw 12')"), ('data/raw_mrdga/neuer_trial\\20240201_UVV_ALLA_LN46o0004_2.csv', "ParserError('Expected 3 fields in line 608, saw 8')"), ('data/raw_mrdga/neuer_trial\\20240201_UVV_ALLA_LN46o0004_3.csv', "ParserError('Expected 3 fields in line 608, saw 8')"), ('data/raw_mrdga/neuer_trial\\20240201_UVV_ALLA_LN46o0004_4.csv', "ParserError('Expected 3 fields in line 608, saw 8')"), ('data/raw_mrdga/neuer_trial\\20240201_UVV_ALLA_LN46o0004_5.csv', "ParserError('Expected 3 fields in line 608, saw 8')"), ('data/raw_mrdga/neuer_trial\\20240201_UVV_ALLA_LN46o0004_6.csv', "ParserError('Expected 3 fields in line 608, saw 12')"), ('data/raw_mrdga/neuer_trial\\20240201_UVV_ALLA_LN46o0004_7.csv', "ParserError('Expected 3 fields in line 608, saw 8')"), ('data/raw_mrdga/neuer_trial\\20240208_UVV_ALLA_LN46o0008_2.csv', "ParserError('Expected 3 fields in line 608, saw 8')"), ('data/raw_mrdga/neuer_trial\\20240208_UVV_ALLA_LN46o0008_3.csv', "ParserError('Expected 3 fields in line 608, saw 8')"), ('data/raw_mrdga/neuer_trial\\20240208_UVV_ALLA_LN46o0008_4.csv', "ParserError('Expected 3 fields in line 608, saw 8')"), ('data/raw_mrdga/neuer_trial\\20240208_UVV_ALLA_LN46o0008_5.csv', "ParserError('Expected 3 fields in line 608, saw 8')"), ('data/raw_mrdga/neuer_trial\\20240208_UVV_ALLA_LN46o0008_7.csv', "ParserError('Expected 3 fields in line 608, saw 8')"), ('data/raw_mrdga/neuer_trial\\20240213_UVV_ALLA_LN55o0004_1.csv', "ParserError('Expected 3 fields in line 608, saw 8')"), ('data/raw_mrdga/neuer_trial\\20240213_UVV_ALLA_LN55o0004_2.csv', "ParserError('Expected 3 fields in line 608, saw 8')"), ('data/raw_mrdga/neuer_trial\\20240213_UVV_ALLA_LN55o0004_3.csv', "ParserError('Expected 3 fields in line 608, saw 8')"), ('data/raw_mrdga/neuer_trial\\20240213_UVV_ALLA_LN55o0004_4.csv', "ParserError('Expected 3 fields in line 608, saw 8')"), ('data/raw_mrdga/neuer_trial\\20240213_UVV_ALLA_LN55o0004_5.csv', "ParserError('Expected 3 fields in line 608, saw 8')"), ('data/raw_mrdga/neuer_trial\\20240213_UVV_ALLA_LN55o0004_6.csv', "ParserError('Expected 3 fields in line 608, saw 10')"), ('data/raw_mrdga/neuer_trial\\20240213_UVV_ALLA_LN55o0004_7.csv', "ParserError('Expected 3 fields in line 608, saw 8')"), ('data/raw_mrdga/neuer_trial\\20240214_UVV_ALLA_LN31o0001_5.csv', "ParserError('Expected 3 fields in line 608, saw 8')"), ('data/raw_mrdga/neuer_trial\\20240214_UVV_ALLA_LN31o0001_6.csv', "ParserError('Expected 3 fields in line 608, saw 8')"), ('data/raw_mrdga/neuer_trial\\20240214_UVV_ALLA_LN31o0005_2.csv', "ParserError('Expected 3 fields in line 608, saw 8')"), ('data/raw_mrdga/neuer_trial\\20240214_UVV_ALLA_LN31o0005_3.csv', "ParserError('Expected 3 fields in line 608, saw 8')"), ('data/raw_mrdga/neuer_trial\\20240214_UVV_ALLA_LN31o0005_4.csv', "ParserError('Expected 3 fields in line 608, saw 8')"), ('data/raw_mrdga/neuer_trial\\20240214_UVV_ALLA_LN31o0005_5.csv', "ParserError('Expected 3 fields in line 608, saw 8')"), ('data/raw_mrdga/neuer_trial\\20240215_UVV_ALLA_LN46o0012_1.csv', "ParserError('Expected 3 fields in line 608, saw 8')"), ('data/raw_mrdga/neuer_trial\\20240215_UVV_ALLA_LN46o0012_2.csv', "ParserError('Expected 3 fields in line 608, saw 8')"), ('data/raw_mrdga/neuer_trial\\20240215_UVV_ALLA_LN46o0012_3.csv', "ParserError('Expected 3 fields in line 608, saw 8')"), ('data/raw_mrdga/neuer_trial\\20240215_UVV_ALLA_LN46o0012_4.csv', "ParserError('Expected 3 fields in line 608, saw 8')"), ('data/raw_mrdga/neuer_trial\\20240215_UVV_ALLA_LN46o0012_5.csv', "ParserError('Expected 3 fields in line 608, saw 8')"), ('data/raw_mrdga/neuer_trial\\20240215_UVV_ALLA_LN46o0012_6.csv', "ParserError('Expected 3 fields in line 608, saw 10')"), ('data/raw_mrdga/neuer_trial\\20240215_UVV_ALLA_LN46o0012_7.csv', "ParserError('Expected 3 fields in line 608, saw 8')"), ('data/raw_mrdga/neuer_trial\\20240220_UVV_ALLA_LN31o0009_1.csv', "ParserError('Expected 3 fields in line 608, saw 8')"), ('data/raw_mrdga/neuer_trial\\20240220_UVV_ALLA_LN31o0009_2.csv', "ParserError('Expected 3 fields in line 608, saw 16')"), ('data/raw_mrdga/neuer_trial\\20240220_UVV_ALLA_LN31o0009_3.csv', "ParserError('Expected 3 fields in line 608, saw 8')"), ('data/raw_mrdga/neuer_trial\\20240220_UVV_ALLA_LN31o0009_4.csv', "ParserError('Expected 3 fields in line 608, saw 14')"), ('data/raw_mrdga/neuer_trial\\20240220_UVV_ALLA_LN31o0009_5.csv', "ParserError('Expected 3 fields in line 608, saw 8')"), ('data/raw_mrdga/neuer_trial\\20240220_UVV_ALLA_LN31o0009_6.csv', "ParserError('Expected 3 fields in line 608, saw 8')"), ('data/raw_mrdga/neuer_trial\\20240220_UVV_ALLA_LN31o0009_8.csv', "ParserError('Expected 3 fields in line 608, saw 8')"), ('data/raw_mrdga/neuer_trial\\20240221_UVV_ALLA_LN31o0010_3.csv', "ParserError('Expected 3 fields in line 734, saw 8')"), ('data/raw_mrdga/neuer_trial\\20240221_UVV_ALLA_LN31o0013_1.csv', "ParserError('Expected 3 fields in line 608, saw 8')"), ('data/raw_mrdga/neuer_trial\\20240221_UVV_ALLA_LN31o0013_3.csv', "ParserError('Expected 3 fields in line 608, saw 8')"), ('data/raw_mrdga/neuer_trial\\20240228_UVV_ALLA_BLPHIo0008_7.csv', "ParserError('Expected 3 fields in line 608, saw 10')")]

Total files checked: 1138
OK (pandas read): 1095
Pandas-only errors: [('../../data/raw_mrdga/neuer_trial\\20240201_UVV_ALLA_LN46o0004_1.csv', "ParserError('Expected 3 fields in line 608, saw 12')"), ('../../data/raw_mrdga/neuer_trial\\20240201_UVV_ALLA_LN46o0004_2.csv', "ParserError('Expected 3 fields in line 608, saw 8')"), ('../../data/raw_mrdga/neuer_trial\\20240201_UVV_ALLA_LN46o0004_3.csv', "ParserError('Expected 3 fields in line 608, saw 8')"), ('../../data/raw_mrdga/neuer_trial\\20240201_UVV_ALLA_LN46o0004_4.csv', "ParserError('Expected 3 fields in line 608, saw 8')"), ('../../data/raw_mrdga/neuer_trial\\20240201_UVV_ALLA_LN46o0004_5.csv', "ParserError('Expected 3 fields in line 608, saw 8')"), ('../../data/raw_mrdga/neuer_trial\\20240201_UVV_ALLA_LN46o0004_6.csv', "ParserError('Expected 3 fields in line 608, saw 12')"), ('../../data/raw_mrdga/neuer_trial\\20240201_UVV_ALLA_LN46o0004_7.csv', "ParserError('Expected 3 fields in line 608, saw 8')"), ('../../data/raw_mrdga/neuer_tri

(['../../data/raw_mrdga/neuer_trial\\20240201_UVV_ALLA_LN46o0001_1.csv',
  '../../data/raw_mrdga/neuer_trial\\20240201_UVV_ALLA_LN46o0001_2.csv',
  '../../data/raw_mrdga/neuer_trial\\20240201_UVV_ALLA_LN46o0001_3.csv',
  '../../data/raw_mrdga/neuer_trial\\20240201_UVV_ALLA_LN46o0001_4.csv',
  '../../data/raw_mrdga/neuer_trial\\20240201_UVV_ALLA_LN46o0001_5.csv',
  '../../data/raw_mrdga/neuer_trial\\20240201_UVV_ALLA_LN46o0001_6.csv',
  '../../data/raw_mrdga/neuer_trial\\20240201_UVV_ALLA_LN46o0001_7.csv',
  '../../data/raw_mrdga/neuer_trial\\20240201_UVV_ALLA_LN46o0002_1.csv',
  '../../data/raw_mrdga/neuer_trial\\20240201_UVV_ALLA_LN46o0002_2.csv',
  '../../data/raw_mrdga/neuer_trial\\20240201_UVV_ALLA_LN46o0002_3.csv',
  '../../data/raw_mrdga/neuer_trial\\20240201_UVV_ALLA_LN46o0002_4.csv',
  '../../data/raw_mrdga/neuer_trial\\20240201_UVV_ALLA_LN46o0002_5.csv',
  '../../data/raw_mrdga/neuer_trial\\20240201_UVV_ALLA_LN46o0002_6.csv',
  '../../data/raw_mrdga/neuer_trial\\20240201_UVV_A

# Try to fix the pandas errors

In [67]:
# replace metadata of files that caused pandas errors with metadata from the model file
# uses existing globals: pandas_errors (list of paths or (path,err)), MODEL_FILE or modelfile

num_re = re.compile(r'^[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?$')

def first_col_value(line):
    parts = line.split(',', 1)
    if not parts:
        return None
    s = parts[0].strip()
    try:
        return float(s)
    except Exception:
        return None


updated = []
skipped = []
for item in pandas_errors:
    fp = item if isinstance(item, str) else item[0]
    try:
        with open(fp, 'r', encoding='utf-8', errors='replace') as f:
            lines = f.readlines()
        # find last data line index in this file
        last_idx = None
        for i, ln in enumerate(lines):
            val = first_col_value(ln)
            if val is not None and 199.5 <= val <= 200.5:
                last_idx = i

        if last_idx is None:
            skipped.append((fp, "no split point found"))
            continue

        # keep data up to last_idx (inclusive), then append model metadata
        out_lines = [ln if ln.endswith('\n') else ln + '\n' for ln in lines[: last_idx + 1]]
        # append model_meta but replace any whole-word "spot1" with this file's first-column header
        first_field = lines[0].rstrip('\n').split(',')[0].strip() or 'spot1'
        for ln in model_meta:
            out_lines.append(re.sub(r'\bspot1\b', first_field, ln) if ln.endswith('\n') else re.sub(r'\bspot1\b', first_field, ln) + '\n')

        # write back (overwrite)
        with open(fp, 'w', encoding='utf-8') as outf:
            outf.writelines(out_lines)
        updated.append(fp)
    except Exception as e:
        skipped.append((fp, repr(e)))

print(f"Updated {len(updated)} files, skipped {len(skipped)} files")
if skipped:
    print("Skipped details (path, reason):")
    for s in skipped:
        print(s)

Updated 43 files, skipped 0 files


# Recheck readability

In [68]:
# check CSV readability in 'data/raw_mrdga/neuer_trial'
target_dir = 'data/raw_mrdga/neuer_trial'
files = glob.glob(os.path.join(target_dir, '*.csv'))

pandas_errors = []   # files pandas could not parse (pandas exception)
csv_fallback_errors = []  # files the csv module also fails to read (likely corrupt)
ok_files = []

for fp in files:
    try:
        # try pandas auto-sep parsing (engine='python' + sep=None uses the C engine's sniffing fallback)
        pd.read_csv(fp, sep=None, engine='python', encoding='utf-8', on_bad_lines='error')
        ok_files.append(fp)
    except Exception as e_pandas:
        # fallback: try Python csv reader to see if file is fundamentally unreadable
        try:
            with open(fp, 'r', encoding='utf-8', errors='replace') as f:
                for _ in csv.reader(f):
                    pass
            pandas_errors.append((fp, repr(e_pandas)))
        except Exception as e_csv:
            csv_fallback_errors.append((fp, repr(e_csv)))

errored_files = sorted({p for p, _ in pandas_errors} | {p for p, _ in csv_fallback_errors})

# summary
print(f"Total files checked: {len(files)}")
print(f"OK (pandas read): {len(ok_files)}")
print(f"Pandas-only errors: {len(pandas_errors)}")
print(f"CSV-module errors (likely unreadable): {len(csv_fallback_errors)}")

# expose lists in the notebook namespace
ok_files, pandas_errors, csv_fallback_errors, errored_files

Total files checked: 1138
OK (pandas read): 1138
Pandas-only errors: 0
CSV-module errors (likely unreadable): 0


(['../../data/raw_mrdga/neuer_trial\\20240201_UVV_ALLA_LN46o0001_1.csv',
  '../../data/raw_mrdga/neuer_trial\\20240201_UVV_ALLA_LN46o0001_2.csv',
  '../../data/raw_mrdga/neuer_trial\\20240201_UVV_ALLA_LN46o0001_3.csv',
  '../../data/raw_mrdga/neuer_trial\\20240201_UVV_ALLA_LN46o0001_4.csv',
  '../../data/raw_mrdga/neuer_trial\\20240201_UVV_ALLA_LN46o0001_5.csv',
  '../../data/raw_mrdga/neuer_trial\\20240201_UVV_ALLA_LN46o0001_6.csv',
  '../../data/raw_mrdga/neuer_trial\\20240201_UVV_ALLA_LN46o0001_7.csv',
  '../../data/raw_mrdga/neuer_trial\\20240201_UVV_ALLA_LN46o0002_1.csv',
  '../../data/raw_mrdga/neuer_trial\\20240201_UVV_ALLA_LN46o0002_2.csv',
  '../../data/raw_mrdga/neuer_trial\\20240201_UVV_ALLA_LN46o0002_3.csv',
  '../../data/raw_mrdga/neuer_trial\\20240201_UVV_ALLA_LN46o0002_4.csv',
  '../../data/raw_mrdga/neuer_trial\\20240201_UVV_ALLA_LN46o0002_5.csv',
  '../../data/raw_mrdga/neuer_trial\\20240201_UVV_ALLA_LN46o0002_6.csv',
  '../../data/raw_mrdga/neuer_trial\\20240201_UVV_A

# copy .yaml files into new folder

In [55]:
target_dir = 'data/raw_mrdga/neuer_trial'
files = glob.glob(os.path.join(target_dir, '*.csv'))
yamlfolder = 'data/raw_mrdga/'
yamlfiles = glob.glob(os.path.join(yamlfolder, '*.yaml'))

import shutil

os.makedirs(target_dir, exist_ok=True)

copied = []
for yf in yamlfiles:
    try:
        dst = os.path.join(target_dir, os.path.basename(yf))
        shutil.copy2(yf, dst)
        copied.append(dst)
    except Exception as e:
        print(f"Failed to copy {yf}: {e}")

print(f"Copied {len(copied)} YAML files to {target_dir}")





Copied 1151 YAML files to ../../data/raw_mrdga/neuer_trial
